# Group 3 — Project 3 (music vs no music: Delsys + force plates + OptiTrack)

Multimodal capture with **music/no-music blocks**. Two takes with the order counterbalanced (Trial 1: music→none then none→music; Trial 2: the reverse).

**Sensors** — Delsys `Ch1` = R tibia, `Ch2` = R (side) leg, `Ch3` = L (side) leg, `Ch4` = L tibia (each EMG + accel); `Ch12` = EKG; `Ch13` = **analog SYNC** (Motive gate). Force plates (2 takes) + OptiTrack markers (L/R foot, L/R shoulder) are on the **Motive clock**.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data (take 001)

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
def get(p): return hf_hub_download(REPO, p, repo_type="dataset")
delsys_csv = get("group_3_3/delsys/Trial_1.csv")                       # Trial_2 = the other order
fp1_csv    = get("group_3_3/bertec/group3_3_001_forceplate_1.csv")
fp2_csv    = get("group_3_3/bertec/group3_3_001_forceplate_2.csv")
markers_csv= get("group_3_3/ot/group3_3_001.csv")
print("downloaded")

### 2 · Load each modality (raw, own clock)
**Delsys** — leg EMG + EKG:

In [ ]:
import delsys, numpy as np, matplotlib.pyplot as plt
lf = delsys.Log(delsys_csv); print("sensors:", lf.sensor_numbers)
rtib = lf.find(sensor_number=1, as_="sensor")[0].emg          # R tibia
plt.figure(figsize=(12,2.5)); plt.plot(rtib.t, rtib()); plt.xlim(30,35); plt.title("R-tibia EMG (Delsys clock)"); plt.show()

**Force plates** — `MotiveLog` (pasted). `MocapTime` is the Motive clock.

In [ ]:
import csv as _csv, numpy as np, pandas as pd, pysampled
class MotiveLog:
    """Load a Motive force-plate 'Device export' CSV. MocapTime is the OptiTrack clock."""
    def __init__(self, fname):
        hdr = {}
        with open(fname, newline='') as f:
            for rc, row in enumerate(_csv.reader(f)):
                if not row: continue
                if row[0] == 'MocapFrame': break
                v = row[1] if len(row) > 1 else None
                try: hdr[row[0]] = float(v)
                except (TypeError, ValueError): hdr[row[0]] = v
        self.data = pd.read_csv(fname, skiprows=rc).rename(columns=lambda x: x.strip())
        self.sr = hdr['Mocap Rate'] * hdr.get('Mocap Rate Multiple', 1)
    def __getattr__(self, k):
        if k in ('Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz','Tz'):
            return pysampled.Data(self.data[k].values, sr=self.sr)
        raise AttributeError(k)

In [ ]:
fp1 = MotiveLog(fp1_csv); fp2 = MotiveLog(fp2_csv)
fz1 = np.abs(np.asarray(fp1.Fz()).ravel()); fp_t = np.arange(len(fz1))/fp1.sr
plt.figure(figsize=(12,2.5)); plt.plot(fp_t, fz1); plt.xlim(30,35); plt.title("force plate 1 |Fz| (Motive clock)"); plt.show()

**OptiTrack markers** — L/R foot, L/R shoulder (Motive clock):

In [ ]:
import pandas as pd, csv as _c
def load_markers(path):
    with open(path) as f:
        h = next(i for i, row in enumerate(_c.reader(f)) if any('Time (Seconds)' in str(x) for x in row))
    m = pd.read_csv(path, skiprows=h); return m.loc[:, ~m.columns.astype(str).str.startswith('Unnamed')]
mk = load_markers(markers_csv); mk_t = mk['Time (Seconds)'].to_numpy(float)
xyz = mk.drop(columns=[c for c in ('Frame','Time (Seconds)') if c in mk]).to_numpy(float)
plt.figure(figsize=(12,2.5)); plt.plot(mk_t, xyz[:, 2]); plt.xlim(30,35); plt.title("first marker, vertical (Motive clock)"); plt.show()

### 3 · Synchronize (Delsys gate) + line signals up

In [ ]:
import numpy as np
def motive_gate_offset(lf):
    """Delsys analog sensor 13, channel 0 = Motive's record gate. Its rising edge is the
    Delsys-clock time when Motive (force plates + markers) began at t=0."""
    a = lf.find(sensor_number=13, as_="sensor")[0].analog
    g = np.asarray(a())[:, 0]; t = np.asarray(a.t)
    return float(t[np.argmax(g > 0.5*(g.min()+g.max()))])

In [ ]:
offset = motive_gate_offset(lf)
print(f"Motive started at Delsys t = {offset:.2f} s")
env = rtib.bandpass(20,450).envelope(); emg_t = np.asarray(env.t) - offset   # onto the Motive clock
fig, ax = plt.subplots(2,1, figsize=(13,5), sharex=True)
ax[0].plot(fp_t, fz1); ax[0].set_ylabel("|Fz| (N)"); ax[0].set_title("force plate (Motive clock)")
ax[1].plot(emg_t, np.asarray(env()).ravel(), color="tab:orange"); ax[1].set_ylabel("R-tibia EMG env")
ax[1].set_xlabel("Motive time (s)"); ax[1].set_title("Delsys EMG shifted onto the Motive clock")
[a.set_xlim(20,40) for a in ax]; plt.tight_layout(); plt.show()

### 4 · Your turn
- Split each take into its **music** and **no-music** blocks (you know the order) and compare — cadence (from the leg **accel**, `.acc`), EMG effort, or heart rate (**EKG** on `sensor_number=12`).
- Legs: `sensor_number` 1–4 (R tibia, R side, L side, L tibia). Markers: L/R foot, L/R shoulder.
- **Take 002** (`Trial_2` + the `...002...` files) has the reversed music order — repeat and compare.